In [1]:
import pandas as pd
import numpy as np
import gc
import json
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

DATA_PROC = Path("../data/processed")
MODELS_DIR = Path("../models")

print("=" * 60)
print("数据文件检查")
print("=" * 60)
for f in ["train_with_all_features.parquet", 
          "item_text_clusters_mpnet.csv", 
          "item_image_clusters.csv"]:
    path = DATA_PROC / f
    assert path.exists(), f"❌ 缺: {f}"
    print(f"  ✅ {f}: {path.stat().st_size/1e6:.1f} MB")

数据文件检查
  ✅ train_with_all_features.parquet: 1744.6 MB
  ✅ item_text_clusters_mpnet.csv: 2.9 MB
  ✅ item_image_clusters.csv: 2.9 MB


In [2]:
%%time
print("⏳ 加载 train_with_all_features.parquet (1.6 GB)...")
df_main = pd.read_parquet(DATA_PROC / "train_with_all_features.parquet")
print(f"   shape: {df_main.shape}")
print(f"   列: {list(df_main.columns)}")

# 拼 mpnet text cluster
text_mpnet = pd.read_csv(
    DATA_PROC / "item_text_clusters_mpnet.csv",
    dtype={'parent_asin': 'str', 'text_cluster_id_mpnet': 'int16'},
)
df_main = df_main.merge(text_mpnet, on='parent_asin', how='left')
print(f"\n   ✅ 拼 text_cluster_mpnet: {df_main.shape}")

# 拼 image cluster (今天的!)
image_clusters = pd.read_csv(
    DATA_PROC / "item_image_clusters.csv",
    dtype={'parent_asin': 'str', 'image_cluster_id': 'int16'},
)
df_main = df_main.merge(image_clusters, on='parent_asin', how='left')
print(f"   ✅ 拼 image_cluster: {df_main.shape}")

# 检查 image_cluster NaN (~180 商品没下载到/损坏)
n_missing = df_main['image_cluster_id'].isnull().sum()
print(f"\n   image_cluster 缺失: {n_missing:,} 行 ({n_missing/len(df_main)*100:.3f}%)")

# 缺失填 -1 (LightGBM 当作特殊类别)
df_main['image_cluster_id'] = df_main['image_cluster_id'].fillna(-1).astype('int16')
print(f"   缺失已填 -1")

print(f"\n最终 shape: {df_main.shape}")
print(f"内存: {df_main.memory_usage(deep=True).sum()/1e9:.2f} GB")

del text_mpnet, image_clusters
gc.collect()

⏳ 加载 train_with_all_features.parquet (1.6 GB)...
   shape: (24653142, 18)
   列: ['user_id', 'parent_asin', 'label', 'user_interaction_count', 'user_avg_rating', 'user_last_timestamp', 'item_interaction_count', 'item_avg_rating', 'item_last_timestamp', 'price', 'price_missing', 'title_length', 'n_categories', 'sub_category_id', 'brand_id', 'user_avg_price', 'user_price_diff', 'pop_x_activity']

   ✅ 拼 text_cluster_mpnet: (24653142, 19)
   ✅ 拼 image_cluster: (24653142, 20)

   image_cluster 缺失: 17,461 行 (0.071%)
   缺失已填 -1

最终 shape: (24653142, 20)
内存: 3.72 GB
CPU times: user 6.35 s, sys: 4.92 s, total: 11.3 s
Wall time: 9.18 s


0

In [3]:
%%time
# 复用 v2/v3 的 random_state=42
train_idx, val_idx = train_test_split(
    df_main.index,
    test_size=0.2,
    random_state=42,
    stratify=df_main['label'],
)
print(f"train: {len(train_idx):,}")
print(f"val:   {len(val_idx):,}")

train: 19,722,513
val:   4,930,629
CPU times: user 4.31 s, sys: 317 ms, total: 4.62 s
Wall time: 4.64 s


In [4]:
%%time

FEATURE_COLS_V4 = [
    # User (3)
    'user_interaction_count', 'user_avg_rating', 'user_last_timestamp',
    # Item (3)
    'item_interaction_count', 'item_avg_rating', 'item_last_timestamp',
    # Item meta (6)
    'price', 'price_missing', 'title_length', 'n_categories',
    'sub_category_id', 'brand_id',
    # Cross (3)
    'user_avg_price', 'user_price_diff', 'pop_x_activity',
    # BERT text cluster (Day 6 最佳 mpnet)
    'text_cluster_id_mpnet',
    # CLIP image cluster (Day 7 新)
    'image_cluster_id',
]
CATEGORICAL_V4 = ['sub_category_id', 'brand_id', 'price_missing', 
                  'text_cluster_id_mpnet', 'image_cluster_id']

print(f"v4 特征数: {len(FEATURE_COLS_V4)}")
print(f"Categorical: {CATEGORICAL_V4}")

X_train = df_main.loc[train_idx, FEATURE_COLS_V4]
y_train = df_main.loc[train_idx, 'label']
X_val = df_main.loc[val_idx, FEATURE_COLS_V4]
y_val = df_main.loc[val_idx, 'label']

print(f"\nX_train: {X_train.shape}")

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=CATEGORICAL_V4)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=CATEGORICAL_V4)

params = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'num_threads': 8,
}

print(f"\n⏳ 训练 v4 (预计 14-16 分钟)...")
print(f"   对比基准: v3-mpnet AUC 0.8122")
print()

model_v4 = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=50),
    ],
)

print(f"\n✅ v4 完成")
print(f"   Best iteration: {model_v4.best_iteration}")
print(f"   Val AUC:    {model_v4.best_score['val']['auc']:.4f}")
print(f"   Val LogLoss:{model_v4.best_score['val']['binary_logloss']:.4f}")
print(f"\n📊 对比:")
print(f"   v3-mpnet (16 特征): AUC 0.8122")
print(f"   v4 (17 特征):       AUC {model_v4.best_score['val']['auc']:.4f}")
print(f"   提升 (image cluster): +{model_v4.best_score['val']['auc'] - 0.8122:.4f}")

model_v4.save_model('../models/lightgbm_v4_clip.txt')
print(f"\n✅ 模型保存: lightgbm_v4_clip.txt")

v4 特征数: 17
Categorical: ['sub_category_id', 'brand_id', 'price_missing', 'text_cluster_id_mpnet', 'image_cluster_id']

X_train: (19722513, 17)

⏳ 训练 v4 (预计 14-16 分钟)...
   对比基准: v3-mpnet AUC 0.8122

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.385708	train's auc: 0.773364	val's binary_logloss: 0.385772	val's auc: 0.773254
[100]	train's binary_logloss: 0.374588	train's auc: 0.788114	val's binary_logloss: 0.374745	val's auc: 0.787791
[150]	train's binary_logloss: 0.369169	train's auc: 0.795772	val's binary_logloss: 0.369417	val's auc: 0.795279
[200]	train's binary_logloss: 0.36536	train's auc: 0.801051	val's binary_logloss: 0.365682	val's auc: 0.800432
[250]	train's binary_logloss: 0.362184	train's auc: 0.805398	val's binary_logloss: 0.362576	val's auc: 0.804683
[300]	train's binary_logloss: 0.360007	train's auc: 0.808265	val's binary_logloss: 0.360463	va

In [5]:
# 特征重要性
import pandas as pd

imp_v4 = pd.DataFrame({
    'feature': FEATURE_COLS_V4,
    'gain': model_v4.feature_importance(importance_type='gain'),
    'split': model_v4.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).reset_index(drop=True)

print("v4 特征重要性 (Gain 排序):")
print(imp_v4.to_string(index=False))

# 找 image_cluster 和 text_cluster 排名
print(f"\nimage_cluster_id 排名: #{imp_v4[imp_v4['feature']=='image_cluster_id'].index[0]+1}/17")
print(f"  gain: {imp_v4[imp_v4['feature']=='image_cluster_id']['gain'].values[0]:,.0f}")
print(f"  split: {imp_v4[imp_v4['feature']=='image_cluster_id']['split'].values[0]:,.0f}")

print(f"\ntext_cluster_id_mpnet 排名: #{imp_v4[imp_v4['feature']=='text_cluster_id_mpnet'].index[0]+1}/17")
print(f"  gain: {imp_v4[imp_v4['feature']=='text_cluster_id_mpnet']['gain'].values[0]:,.0f}")
print(f"  split: {imp_v4[imp_v4['feature']=='text_cluster_id_mpnet']['split'].values[0]:,.0f}")

v4 特征重要性 (Gain 排序):
               feature         gain  split
item_interaction_count 1.138433e+07   1112
   item_last_timestamp 3.845705e+06   1513
   user_last_timestamp 2.915801e+06   1895
       user_price_diff 2.043043e+06    712
        user_avg_price 1.796401e+06   1187
        pop_x_activity 1.760769e+06    471
                 price 1.751058e+06    658
user_interaction_count 1.283738e+06   1274
              brand_id 1.204454e+06   2768
 text_cluster_id_mpnet 8.320017e+05   1344
          title_length 7.838891e+05    355
      image_cluster_id 4.183039e+05   1113
       item_avg_rating 2.450367e+05    270
       sub_category_id 7.739021e+04    111
         price_missing 7.477868e+04     47
       user_avg_rating 5.159297e+04    166
          n_categories 1.805793e+02      4

image_cluster_id 排名: #12/17
  gain: 418,304
  split: 1,113

text_cluster_id_mpnet 排名: #10/17
  gain: 832,002
  split: 1,344


In [6]:
%%time
import numpy as np

# v4 预测
df_main['y_pred_v4'] = np.nan
X_val_v4 = df_main.loc[val_idx, FEATURE_COLS_V4]
df_main.loc[val_idx, 'y_pred_v4'] = model_v4.predict(
    X_val_v4, num_iteration=model_v4.best_iteration
)

df_val_v4 = df_main.loc[val_idx, ['user_id', 'label', 'y_pred_v4']].copy()
df_val_v4.rename(columns={'y_pred_v4': 'y_pred'}, inplace=True)

def compute_user_metrics(group, k_list=[5, 10, 20]):
    sorted_group = group.sort_values('y_pred', ascending=False)
    labels = sorted_group['label'].values
    n_pos = labels.sum()
    n_total = len(labels)
    metrics = {}
    for k in k_list:
        k_actual = min(k, n_total)
        top_k_labels = labels[:k_actual]
        n_hits = top_k_labels.sum()
        metrics[f'precision@{k}'] = n_hits / k_actual if k_actual > 0 else 0
        metrics[f'recall@{k}'] = n_hits / n_pos if n_pos > 0 else np.nan
        gains = (2 ** top_k_labels - 1)
        discounts = 1 / np.log2(np.arange(k_actual) + 2)
        dcg = (gains * discounts).sum()
        n_pos_in_topk = min(n_pos, k_actual)
        ideal_labels = np.zeros(k_actual)
        ideal_labels[:int(n_pos_in_topk)] = 1
        ideal_gains = (2 ** ideal_labels - 1)
        idcg = (ideal_gains * discounts).sum()
        metrics[f'ndcg@{k}'] = dcg / idcg if idcg > 0 else np.nan
    return pd.Series(metrics)

print("⏳ 计算 v4 分用户指标 (~2.5 min)...")
user_metrics_v4 = df_val_v4.groupby('user_id').apply(
    compute_user_metrics, k_list=[5, 10, 20], include_groups=False,
)
summary_v4 = user_metrics_v4.mean()
print(f"✅ v4 完成")
print(summary_v4)

⏳ 计算 v4 分用户指标 (~2.5 min)...
✅ v4 完成
precision@5     0.209853
recall@5        0.894744
ndcg@5          0.770097
precision@10    0.178782
recall@10       0.968765
ndcg@10         0.799958
precision@20    0.169792
recall@20       0.991646
ndcg@20         0.808693
dtype: float64
CPU times: user 4min 17s, sys: 4.2 s, total: 4min 21s
Wall time: 2min 15s


In [7]:
# 完整 v0-v4 对比表
print("=" * 95)
print("🏆 Day 7 完整 6 模型对比")
print("=" * 95)

models_summary = [
    ('v0 baseline',    6,  0.7645, 0.3875, None,           None,           None),
    ('v1 tuned',       6,  0.7738, 0.3810, None,           None,           None),
    ('v2 meta+cross',  15, 0.8100, 0.3582, 0.7990,         0.1787,         0.9686),
    ('v3-MiniLM',      16, 0.8116, 0.3571, 0.8001,         0.1788,         0.9688),
    ('v3-mpnet',       16, 0.8122, 0.3567, 0.8006,         0.1788,         0.9690),
    ('v4 + image',     17, 0.8115, 0.3573, summary_v4['ndcg@10'], summary_v4['precision@10'], summary_v4['recall@10']),
]

print(f"\n{'模型':<18}{'特征':<6}{'Val AUC':<10}{'NDCG@10':<10}{'Prec@10':<10}{'Recall@10':<10}")
print("-" * 70)
for name, n_feat, auc, _, ndcg, prec, rec in models_summary:
    ndcg_str = f"{ndcg:.4f}" if ndcg else "—"
    prec_str = f"{prec:.4f}" if prec else "—"
    rec_str = f"{rec:.4f}" if rec else "—"
    print(f"{name:<18}{n_feat:<6}{auc:.4f}    {ndcg_str:<10}{prec_str:<10}{rec_str:<10}")

print(f"\n📊 v4 vs v3-mpnet (image cluster 边际效益):")
print(f"  AUC:      {summary_v4['ndcg@10']:.4f} vs 0.8006  ← 加图片反而下降!")
print(f"  NDCG@10:  {summary_v4['ndcg@10']:.4f} vs 0.8006")
print(f"  Prec@10:  {summary_v4['precision@10']:.4f} vs 0.1788")
print(f"  Recall@10: {summary_v4['recall@10']:.4f} vs 0.9690")

🏆 Day 7 完整 6 模型对比

模型                特征    Val AUC   NDCG@10   Prec@10   Recall@10 
----------------------------------------------------------------------
v0 baseline       6     0.7645    —         —         —         
v1 tuned          6     0.7738    —         —         —         
v2 meta+cross     15    0.8100    0.7990    0.1787    0.9686    
v3-MiniLM         16    0.8116    0.8001    0.1788    0.9688    
v3-mpnet          16    0.8122    0.8006    0.1788    0.9690    
v4 + image        17    0.8115    0.8000    0.1788    0.9688    

📊 v4 vs v3-mpnet (image cluster 边际效益):
  AUC:      0.8000 vs 0.8006  ← 加图片反而下降!
  NDCG@10:  0.8000 vs 0.8006
  Prec@10:  0.1788 vs 0.1788
  Recall@10: 0.9688 vs 0.9690


In [8]:
import json

day7_report = {
    'date': '2026-05-26',
    'day': 'Day 7',
    'experiment_type': 'CLIP image embedding multimodal',
    'result': 'NEGATIVE - image_cluster led to AUC degradation',
    
    'models': {
        'v3_mpnet_baseline': {'features': 16, 'val_auc': 0.8122},
        'v4_with_image':     {'features': 17, 'val_auc': float(model_v4.best_score['val']['auc'])},
        'auc_delta': float(model_v4.best_score['val']['auc']) - 0.8122,
    },
    
    'feature_importance_v4_top5': imp_v4.head(5).to_dict(orient='records'),
    
    'image_cluster_diagnosis': {
        'rank': 12,
        'gain': float(imp_v4[imp_v4['feature']=='image_cluster_id']['gain'].values[0]),
        'gain_vs_text': '50% of text_cluster gain (832K)',
    },
    
    'text_cluster_impact_after_image': {
        'v3_gain': 950432,
        'v4_gain': float(imp_v4[imp_v4['feature']=='text_cluster_id_mpnet']['gain'].values[0]),
        'change': '-12%, signal dispersed across two cluster features',
    },
    
    'key_findings': {
        '1_signal_redundancy': 'CLIP image signal mostly covered by sub_category + text_cluster',
        '2_feature_interference': 'Adding redundant feature dispersed text_cluster signal by 12%',
        '3_mild_overfitting': 'train-val gap expanded from -0.0005 to -0.0012',
        '4_production_decision': 'Do NOT use CLIP image cluster in production - net negative',
    },
    
    'engineering_metrics': {
        'images_downloaded': 207207,
        'download_success_rate': '99.91%',
        'clip_extract_time_min': 5.6,
        'clip_throughput_img_per_sec': 619,
        'gpu_cost_yuan': '~10-12',
    },
    
    'next_steps': [
        'Day 8: Try alternative image strategy (CLIP text-image cosine instead of cluster)',
        'Day 8: Or skip image, focus on user behavior sequence (DIN-style)',
        'Day 9: Resume model architecture exploration (DeepFM/DIN)',
    ],
}

with open('../models/day7_v4_report.json', 'w') as f:
    json.dump(day7_report, f, indent=2, ensure_ascii=False, default=str)

model_v4.save_model('../models/lightgbm_v4_clip.txt')

print("✅ Day 7 完整报告已保存")
print(f"   Val AUC: {model_v4.best_score['val']['auc']:.4f}")
print(f"   关键结论: image_cluster 边际收益 -0.0007 (净负面)")

✅ Day 7 完整报告已保存
   Val AUC: 0.8115
   关键结论: image_cluster 边际收益 -0.0007 (净负面)


In [9]:
%%time

# v4-a = v3 - text_cluster_mpnet + image_cluster_id
# 验证: image_cluster 是"强但冗余"还是"弱且无效"

FEATURE_COLS_V4A = [
    # User 3
    'user_interaction_count', 'user_avg_rating', 'user_last_timestamp',
    # Item 3
    'item_interaction_count', 'item_avg_rating', 'item_last_timestamp',
    # Item meta 6
    'price', 'price_missing', 'title_length', 'n_categories',
    'sub_category_id', 'brand_id',
    # Cross 3
    'user_avg_price', 'user_price_diff', 'pop_x_activity',
    # ❌ 移除 text_cluster_id_mpnet
    # ✅ 保留 image_cluster_id (替换文本)
    'image_cluster_id',
]
CATEGORICAL_V4A = ['sub_category_id', 'brand_id', 'price_missing', 'image_cluster_id']

print(f"v4-a 特征数: {len(FEATURE_COLS_V4A)} (替换实验)")
print(f"  保留 image_cluster_id (CLIP)")
print(f"  移除 text_cluster_id_mpnet (BERT)")
print(f"Categorical: {CATEGORICAL_V4A}")

X_train_a = df_main.loc[train_idx, FEATURE_COLS_V4A]
y_train_a = df_main.loc[train_idx, 'label']
X_val_a = df_main.loc[val_idx, FEATURE_COLS_V4A]
y_val_a = df_main.loc[val_idx, 'label']

print(f"\nX_train: {X_train_a.shape}")

train_data_a = lgb.Dataset(X_train_a, label=y_train_a, categorical_feature=CATEGORICAL_V4A)
val_data_a = lgb.Dataset(X_val_a, label=y_val_a, reference=train_data_a, categorical_feature=CATEGORICAL_V4A)

params = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'num_threads': 8,
}

print(f"\n⏳ 训练 v4-a (预计 13-15 分钟)...")
print(f"   对比基准:")
print(f"     v3-mpnet (text only):     AUC 0.8122")
print(f"     v4 (text + image):        AUC 0.8115 (-0.0007)")
print(f"     v4-a (image only):        AUC ???? (即将)")
print()

model_v4a = lgb.train(
    params,
    train_data_a,
    num_boost_round=500,
    valid_sets=[train_data_a, val_data_a],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=50),
    ],
)

v4a_auc = model_v4a.best_score['val']['auc']
v4a_logloss = model_v4a.best_score['val']['binary_logloss']

print(f"\n✅ v4-a 完成")
print(f"   Best iteration: {model_v4a.best_iteration}")
print(f"   Val AUC:    {v4a_auc:.4f}")
print(f"   Val LogLoss: {v4a_logloss:.4f}")

print(f"\n📊 3 方对比:")
print(f"   v3-mpnet  (text only):  AUC 0.8122  ← 基准")
print(f"   v4-a      (image only): AUC {v4a_auc:.4f}  ← 仅图片!")
print(f"   v4        (text+image): AUC 0.8115")

print(f"\n🔍 关键诊断:")
delta_a_vs_v3 = v4a_auc - 0.8122
if delta_a_vs_v3 > -0.005:
    print(f"   v4-a vs v3-mpnet: {delta_a_vs_v3:+.4f}")
    print(f"   → 图片信号 ≈ 文本信号 (强度相当)")
    print(f"   → 那么 v4 下降 = 信号冗余 + 分散")
elif delta_a_vs_v3 < -0.010:
    print(f"   v4-a vs v3-mpnet: {delta_a_vs_v3:+.4f}")
    print(f"   → 图片信号 << 文本信号 (本身就弱)")
    print(f"   → 那么 v4 下降 = 弱信号 + 干扰")
else:
    print(f"   v4-a vs v3-mpnet: {delta_a_vs_v3:+.4f}")
    print(f"   → 图片信号 < 文本信号 (中等)")
    print(f"   → v4 下降 = 弱信号 + 部分冗余")

model_v4a.save_model('../models/lightgbm_v4a_image_only.txt')
print(f"\n✅ 模型保存: lightgbm_v4a_image_only.txt")

v4-a 特征数: 16 (替换实验)
  保留 image_cluster_id (CLIP)
  移除 text_cluster_id_mpnet (BERT)
Categorical: ['sub_category_id', 'brand_id', 'price_missing', 'image_cluster_id']

X_train: (19722513, 16)

⏳ 训练 v4-a (预计 13-15 分钟)...
   对比基准:
     v3-mpnet (text only):     AUC 0.8122
     v4 (text + image):        AUC 0.8115 (-0.0007)
     v4-a (image only):        AUC ???? (即将)

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.385104	train's auc: 0.774832	val's binary_logloss: 0.385188	val's auc: 0.774663
[100]	train's binary_logloss: 0.374376	train's auc: 0.788533	val's binary_logloss: 0.374553	val's auc: 0.788155
[150]	train's binary_logloss: 0.368867	train's auc: 0.796163	val's binary_logloss: 0.369128	val's auc: 0.795632
[200]	train's binary_logloss: 0.364631	train's auc: 0.801988	val's binary_logloss: 0.364967	val's auc: 0.801341
[250]	train's binary_logloss: 0.36259

In [10]:
%%time
import numpy as np

# v4-a 预测
print("⏳ 计算 v4-a 预测...")
df_main['y_pred_v4a'] = np.nan
X_val_v4a = df_main.loc[val_idx, FEATURE_COLS_V4A]
df_main.loc[val_idx, 'y_pred_v4a'] = model_v4a.predict(
    X_val_v4a, num_iteration=model_v4a.best_iteration
)

# 我们已经有 v4 的预测 (上一个 cell 算过)
# 检查
assert 'y_pred_v4' in df_main.columns, "需要先跑 v4 预测 cell"
print(f"✅ y_pred_v4a 已计算: {df_main['y_pred_v4a'].notna().sum():,}")
print(f"✅ y_pred_v4 已存在: {df_main['y_pred_v4'].notna().sum():,}")

⏳ 计算 v4-a 预测...
✅ y_pred_v4a 已计算: 4,930,629
✅ y_pred_v4 已存在: 4,930,629
CPU times: user 2min 26s, sys: 1.62 s, total: 2min 27s
Wall time: 22.5 s


In [11]:
# 检查是否有 v3-mpnet 模型
import os
v3_path = '../models/lightgbm_v3_mpnet.txt'
print(f"v3-mpnet 模型文件: {v3_path}")
print(f"存在: {os.path.exists(v3_path)}")

if os.path.exists(v3_path):
    print("✅ 加载 v3-mpnet, 计算预测...")
    model_v3 = lgb.Booster(model_file=v3_path)
    
    FEATURE_COLS_V3 = [c for c in FEATURE_COLS_V4 if c != 'image_cluster_id']
    X_val_v3 = df_main.loc[val_idx, FEATURE_COLS_V3]
    df_main['y_pred_v3'] = np.nan
    df_main.loc[val_idx, 'y_pred_v3'] = model_v3.predict(X_val_v3)
    print(f"✅ y_pred_v3 已计算")
else:
    print("⚠️ 没有 v3-mpnet 模型文件, 跳过 (我们会用 0.8122 作为参考)")

v3-mpnet 模型文件: ../models/lightgbm_v3_mpnet.txt
存在: True
✅ 加载 v3-mpnet, 计算预测...
✅ y_pred_v3 已计算


In [12]:
%%time
print("⏳ 计算每个 sub_category 的 NDCG (3 模型)...")

# 准备数据
val_df = df_main.loc[val_idx, [
    'user_id', 'parent_asin', 'sub_category_id', 'label',
    'y_pred_v4', 'y_pred_v4a',
]].copy()

if 'y_pred_v3' in df_main.columns:
    val_df['y_pred_v3'] = df_main.loc[val_idx, 'y_pred_v3'].values

# 简化版: 每个 sub_category, 每个用户算 NDCG@10
def ndcg_at_k(group, pred_col, k=10):
    """简化 NDCG"""
    sorted_group = group.sort_values(pred_col, ascending=False)
    labels = sorted_group['label'].values
    n_pos = labels.sum()
    if n_pos == 0:
        return np.nan
    k_actual = min(k, len(labels))
    top_k = labels[:k_actual]
    discounts = 1 / np.log2(np.arange(k_actual) + 2)
    dcg = (top_k * discounts).sum()
    ideal_labels = np.zeros(k_actual)
    ideal_labels[:int(min(n_pos, k_actual))] = 1
    idcg = (ideal_labels * discounts).sum()
    return dcg / idcg if idcg > 0 else np.nan

# 按 sub_category 分组, 算每个模型的 NDCG (太慢, 简化用 AUC)
print("使用 AUC@sub_category 替代 (更快)")
from sklearn.metrics import roc_auc_score

results = []
for cat_id in val_df['sub_category_id'].unique():
    if cat_id < 0:
        continue
    cat_df = val_df[val_df['sub_category_id'] == cat_id]
    n_samples = len(cat_df)
    if n_samples < 100 or cat_df['label'].nunique() < 2:
        continue
    
    row = {
        'sub_category_id': cat_id,
        'n_samples': n_samples,
        'n_positive': int(cat_df['label'].sum()),
        'auc_v4': roc_auc_score(cat_df['label'], cat_df['y_pred_v4']),
        'auc_v4a': roc_auc_score(cat_df['label'], cat_df['y_pred_v4a']),
    }
    if 'y_pred_v3' in val_df.columns:
        row['auc_v3'] = roc_auc_score(cat_df['label'], cat_df['y_pred_v3'])
    results.append(row)

results_df = pd.DataFrame(results).sort_values('n_samples', ascending=False)

# 计算 deltas
if 'auc_v3' in results_df.columns:
    results_df['v4_vs_v3'] = results_df['auc_v4'] - results_df['auc_v3']
    results_df['v4a_vs_v3'] = results_df['auc_v4a'] - results_df['auc_v3']

print(f"\n分析完成: {len(results_df)} 个 sub_category")
print(f"\nTop 10 (按样本量):")
print(results_df.head(10).round(4).to_string(index=False))

⏳ 计算每个 sub_category 的 NDCG (3 模型)...
使用 AUC@sub_category 替代 (更快)

分析完成: 28 个 sub_category

Top 10 (按样本量):
 sub_category_id  n_samples  n_positive  auc_v4  auc_v4a  auc_v3  v4_vs_v3  v4a_vs_v3
               0    1261428      199832  0.8169   0.8173  0.8172   -0.0003     0.0000
               1    1099065      186018  0.8061   0.8061  0.8066   -0.0005    -0.0005
               4     669347      110966  0.8176   0.8179  0.8187   -0.0011    -0.0008
               2     601372      110396  0.8043   0.8042  0.8047   -0.0004    -0.0005
               3     527600       84881  0.8107   0.8108  0.8119   -0.0012    -0.0011
               6     268497       47855  0.8147   0.8151  0.8159   -0.0012    -0.0008
               5     202239       32074  0.7978   0.7984  0.7987   -0.0009    -0.0003
               7     183981       29445  0.8271   0.8270  0.8270    0.0001    -0.0000
               8      66805       11746  0.7738   0.7728  0.7746   -0.0008    -0.0018
               9      22458       

In [13]:
# 计算 "v4-a 比 v3 强 / 差" 的类目分布
print("=" * 80)
print("🏆 v4-a (image only) vs v3-mpnet (text only) 在 28 个类目的表现分布")
print("=" * 80)

if 'v4a_vs_v3' in results_df.columns:
    v4a_wins = (results_df['v4a_vs_v3'] > 0).sum()
    v4a_ties = (results_df['v4a_vs_v3'].abs() <= 0.0003).sum()
    v4a_losses = (results_df['v4a_vs_v3'] < -0.0003).sum()
    
    print(f"\nv4-a (CLIP) vs v3-mpnet (BERT):")
    print(f"  CLIP 赢:  {v4a_wins:>2} 个类目")
    print(f"  打平:    {v4a_ties:>2} 个类目  (|delta| <= 0.0003)")
    print(f"  BERT 赢:  {v4a_losses:>2} 个类目")

print(f"\n{'='*80}")
print("📊 各类目 v4 (叠加) vs v3 (单独) 的影响")
print(f"{'='*80}")

if 'v4_vs_v3' in results_df.columns:
    v4_wins = (results_df['v4_vs_v3'] > 0).sum()
    v4_ties = (results_df['v4_vs_v3'].abs() <= 0.0003).sum()
    v4_losses = (results_df['v4_vs_v3'] < -0.0003).sum()
    
    print(f"\nv4 (text+image) vs v3-mpnet (text only):")
    print(f"  v4 涨:  {v4_wins:>2} 个类目")
    print(f"  打平:  {v4_ties:>2} 个类目  (|delta| <= 0.0003)")
    print(f"  v4 跌:  {v4_losses:>2} 个类目  ← 大多数!")

# 找 v4-a 涨最多 / 跌最多的类目
print(f"\n{'='*80}")
print("🎯 v4-a (CLIP) 表现最好的 5 个类目")
print(f"{'='*80}")
top_clip = results_df.sort_values('v4a_vs_v3', ascending=False).head(5)
print(top_clip[['sub_category_id', 'n_samples', 'auc_v3', 'auc_v4a', 'v4a_vs_v3']].round(4).to_string(index=False))

print(f"\n{'='*80}")
print("🎯 v4-a (CLIP) 表现最差的 5 个类目")
print(f"{'='*80}")
bottom_clip = results_df.sort_values('v4a_vs_v3', ascending=True).head(5)
print(bottom_clip[['sub_category_id', 'n_samples', 'auc_v3', 'auc_v4a', 'v4a_vs_v3']].round(4).to_string(index=False))

# 关键统计
print(f"\n{'='*80}")
print("📐 最终洞察")
print(f"{'='*80}")
print(f"\n核心发现:")
print(f"  1. v4-a (CLIP only) AUC = 0.8117, 仅比 v3-mpnet (0.8122) 差 0.0005")
print(f"     → 图片和文本信号强度几乎相当!")
print(f"  2. v4 (text+image) AUC = 0.8115, 比 v3 和 v4-a 都差")
print(f"     → 叠加 = dominated strategy, 信号严重冗余")
print(f"  3. Hair Care 类目: image (0.8173) 甚至略胜 text (0.8172)")
print(f"     → 不同类目最优 modality 不同, 暗示 dynamic feature selection")

🏆 v4-a (image only) vs v3-mpnet (text only) 在 28 个类目的表现分布

v4-a (CLIP) vs v3-mpnet (BERT):
  CLIP 赢:  12 个类目
  打平:     5 个类目  (|delta| <= 0.0003)
  BERT 赢:  13 个类目

📊 各类目 v4 (叠加) vs v3 (单独) 的影响

v4 (text+image) vs v3-mpnet (text only):
  v4 涨:   9 个类目
  打平:   2 个类目  (|delta| <= 0.0003)
  v4 跌:  18 个类目  ← 大多数!

🎯 v4-a (CLIP) 表现最好的 5 个类目
 sub_category_id  n_samples  auc_v3  auc_v4a  v4a_vs_v3
              25        112  0.6019   0.6280     0.0262
              20        112  0.8205   0.8376     0.0171
              23        182  0.7235   0.7394     0.0159
              14        123  0.6220   0.6368     0.0149
              12        151  0.7401   0.7453     0.0053

🎯 v4-a (CLIP) 表现最差的 5 个类目
 sub_category_id  n_samples  auc_v3  auc_v4a  v4a_vs_v3
              19        138  0.6995   0.6852    -0.0144
              13        736  0.7736   0.7641    -0.0095
              21        270  0.7533   0.7449    -0.0084
              16       1186  0.7808   0.7761    -0.0048
              28   